# CineScope Phase 2 — CenterNet face detector on WIDER FACE

Settings (right-hand panel): **Accelerator: GPU** (T4 or P100), **Internet: on**
(needs a phone-verified Kaggle account). Inputs: your private `cinescope-code` dataset.

Run it with **Save Version → Save & Run All (Commit)**: it then runs in the
background (up to 12 h) and everything in `/kaggle/working` is kept as the
version's output. The first epoch prints how long it took: 60 epochs have to fit in
the 12-hour limit (lower `--epochs` if they don't, or let it resume in a second version).

If a run stops early, create a new version with the previous version's output
added as an input: the training cell resumes from the last saved epoch.

In [ ]:
# 1. Find the code: the "cinescope-code" dataset (Kaggle unpacks the zip) -> a writable copy
import glob, os, shutil, subprocess, sys
hits = glob.glob("/kaggle/input/**/p2_detection/train_faces.py", recursive=True)
assert hits, "Add the cinescope-code dataset to this notebook (Add Input -> Your Datasets)"
src = os.path.dirname(os.path.dirname(hits[0]))
shutil.copytree(src, "/kaggle/working/cinescope", dirs_exist_ok=True)
os.chdir("/kaggle/working/cinescope")
print("code from", src)
!nvidia-smi --query-gpu=name,memory.total --format=csv
!df -h /kaggle/working /tmp | tail -n 2

In [ ]:
# 2. WIDER FACE: use an attached copy if there is one, otherwise download (~1.8 GB for train + val)
import pathlib
def find(name):
    h = glob.glob(f"/kaggle/input/**/{name}", recursive=True)
    return h[0] if h else None

gt_txt = find("wider_face_train_bbx_gt.txt")
if gt_txt and find("WIDER_train"):
    WIDER = str(pathlib.Path(gt_txt).parents[1])
    print("using attached WIDER FACE at", WIDER)
else:
    WIDER = "/tmp/wider"
    os.makedirs(WIDER, exist_ok=True)
    hf = "https://huggingface.co/datasets/CUHK-CSE/wider_face/resolve/main/data"
    for z in ["WIDER_train.zip", "WIDER_val.zip", "wider_face_split.zip"]:
        subprocess.run(["wget", "-q", "-L", "-O", f"{WIDER}/{z}", f"{hf}/{z}"], check=True)
        subprocess.run(["unzip", "-q", "-o", f"{WIDER}/{z}", "-d", WIDER], check=True)
        os.remove(f"{WIDER}/{z}")
# official evaluation toolkit (ground-truth .mat files for Easy / Medium / Hard)
EVAL_TOOLS = find("wider_easy_val.mat")
if EVAL_TOOLS:
    EVAL_TOOLS = os.path.dirname(EVAL_TOOLS)
else:
    subprocess.run(["wget", "-q", "-O", "/tmp/eval_tools.zip",
                    "http://shuoyang1213.me/WIDERFACE/support/eval_script/eval_tools.zip"], check=True)
    subprocess.run(["unzip", "-q", "-o", "/tmp/eval_tools.zip", "-d", "/tmp"], check=True)
    EVAL_TOOLS = os.path.dirname(glob.glob("/tmp/**/wider_easy_val.mat", recursive=True)[0])
print(WIDER, os.listdir(WIDER), EVAL_TOOLS)

In [ ]:
# 3. Tests + 1-minute smoke run (fails fast if anything is wrong before the long run)
!pip -q install pytest
!python -m pytest -q tests/test_week6.py tests/test_week7.py tests/test_week8.py
!python p2_detection/train_faces.py --synthetic --epochs 1 --out /tmp/smoke --ckpt-dir /tmp/smoke

In [ ]:
# 4. Train (resumes automatically from an earlier version's checkpoint if one is attached)
prev = glob.glob("/kaggle/input/**/checkpoints/centernet_r18.pt", recursive=True)
os.makedirs("/kaggle/working/checkpoints", exist_ok=True)
if prev:
    shutil.copy(prev[0], "/kaggle/working/checkpoints/centernet_r18.pt")
    print("resuming from", prev[0])
!python p2_detection/train_faces.py --wider {WIDER} --epochs 60 --bs 32 --workers 4 \
    --ckpt-dir /kaggle/working/checkpoints --out /kaggle/working/results --resume

In [ ]:
# 5. Official Easy / Medium / Hard evaluation on WIDER val
!python p2_detection/eval_wider.py --wider {WIDER} --ckpt /kaggle/working/checkpoints/centernet_r18.pt \
    --eval-tools {EVAL_TOOLS} --out /kaggle/working/results

In [ ]:
# 6. Bundle what to bring home: results (small) and the checkpoint (~70 MB)
!cd /kaggle/working && zip -qr phase2_results.zip results checkpoints && ls -lh phase2_results.zip